In [67]:
import numpy as np
import random
from tqdm.auto import tqdm

In [83]:
problem = np.load('problem_2.npz')
x = problem['x']
y = problem['y']

# convert normal python array into numpy ndarray
# x = np.array(x)

PROBLEM_SIZE  = np.shape(x)[0]
PROBLEM_SIZE

3

In [69]:
max_coefficient = 10 # adjustable depending on output scale

def compute_coefficient():
    """returns a random coefficient between 1 and maximum coefficient"""
    return np.random.randint(1, max_coefficient)
    



def are_compatible(operator, value, base=0):
    match operator:
        case "/":
            if isinstance(value, int) or isinstance(value, float):
                return False if value == 0 else True
            else : # check if all values are non-zero
                return False if (0 in value) else True
            
        case "^":
            if not isinstance(value, np.ndarray) and not isinstance(base, np.ndarray): # (non array) ^ (non array)
                return False if base < 0 and not isinstance(value, int) else True
            elif not isinstance(value, np.ndarray) and isinstance(base, np.ndarray): # array ^ (non array)
                return False if np.any(base < 0) and not isinstance(value, int) else True
            elif isinstance(value, np.ndarray) and not isinstance(base, np.ndarray): # (non array) ^ array
                return False if base < 0 and any((not isinstance(i, int)) for i in value) else True
            else : # array ^ array
                for i in range(len(value)):                    
                    if (base[i] < 0 and not isinstance(value[i], int)):
                        return False 
                return True
            
        case "log" :
            if isinstance(value, int) or isinstance(value, float):
                return False if value <= 0 else True
            else : # check if all values are non-negative
                return False if (np.any(value <= 0)) else True
        
        case "arcos" :
            if isinstance(value, int) or isinstance(value, float):
                return False if value < -1 or value > 1 else True
            else : # check if all values are between -1 and 1
                return False if (np.any(value < -1 ) or np.any(value > 1) ) else True
            
        case "arcsin" :
            if isinstance(value, int) or isinstance(value, float):
                return False if value < -1 or value > 1 else True
            else : # check if all values are between -1 and 1
                return False if (np.any(value < -1 )or np.any(value > 1) )else True
        
        case "sqrt" :
            if isinstance(value, int) or isinstance(value, float):
                return False if value < 0 else True
            else : # check if all values are non-negative
                return False if (np.any(value < 0) )else True

        case "reciprocal" :
            if isinstance(value, int) or isinstance(value, float):
                return False if value == 0 else True
            else:
                return False if (np.any(value == 0)) else True
            
        case "tan" :
            if isinstance(value, int) or isinstance(value, float):
                k = (value - np.pi / 2) / np.pi
                return False if k.is_integer() else True
            else:
                for i in range(len(value)):
                    k = (value[i] - np.pi / 2) / np.pi
                    if k.is_integer():
                        return False
                return True



In [70]:
# print(np.divide(5,2))
# print(np.remainder(5,2))
# print(np.pow(5,2))

# a x b = b x 1  ->  a x 1
# 3 x 3 * 3 x 
# y = np.add(np.array(x[0]), 5)
print(y)

# print(np.array(x).shape)


[3, 4, 5]


In [71]:
r = [1, 2, 3]
f = np.array([1,2,3])
print(isinstance(r, np.ndarray))
print(isinstance(f, np.ndarray))

False
True


In [72]:
binary_operators = [np.add, np.subtract, np.dot, np.divide, np.pow]

BINARY_OPERATORS = {
    "+": np.add,
    "-": np.subtract,
    "*": np.dot,
    "/": np.divide,
    "^": np.pow
}

#https://numpy.org/doc/2.1/reference/routines.math.html
UNARY_OPERATORS = {
        "": lambda x: x,  
        "sin": np.sin,
        "cos": np.cos,
        "tan":np.tan,
        "log": np.log,
        "exp": np.exp,
        "arccos": np.arccos,
        "arcsin":np.arcsin,
        "arctan":np.arctan,
        "sqrt":np.sqrt,
        "cbrt":np.cbrt,
        "square":np.square,
        "abs":np.abs,
        "reciprocal":np.reciprocal
    }

#VARIABLES = [f"X_{i}" for i in range(PROBLEM_SIZE)]

#VARIABLES_WEIGHTS = [[1/len(VARIABLES) for _ in range(len(VARIABLES))]]
VARIABLES_MAP = {f"X_{i}": x[i] for i in range(PROBLEM_SIZE)}    # {'X_0': [1, 2, 3], 'X_1': [4, 5, 6], 'X_2': [7, 8, 9]}
LEAVES = [i for i in range(10)] + list(VARIABLES_MAP.keys())
print(VARIABLES_MAP)
print(LEAVES)

{'X_0': array([1, 2, 3]), 'X_1': array([2, 3, 4]), 'X_2': array([3, 4, 5])}
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 'X_0', 'X_1', 'X_2']


## Tree structure

In [80]:
class TreeNode:
    def __init__(self, value):
        self.value = value      # This can be an operator or operand
        self.left = None        # Left child
        self.right = None       # Right child
        self.coefficient = None # multiplicative coefficient for a variable

class Tree:
    def __init__(self, root, depth):
        self.root = root
        self.dept = depth

def validate_tree(node):
    if not node:
        return True
    
    if node.value in BINARY_OPERATORS:
        if not node.left or not node.right:
            return False  # Operators must have two children
        return validate_tree(node.left) and validate_tree(node.right)
    
    elif node.value in UNARY_OPERATORS:  # Allow unary operators
        if node.right and not node.left:
            return validate_tree(node.right) 
        return False  # Unary operators must have one child on the right
    
    # elif node.value in VARIABLES_MAP and isinstance(node.value, str):  # Allow variables
    elif node.value in LEAVES:
        return True
    else:
        return False  # Invalid value
    
    
# (3 + 2) * (4 + 5)        Treenode (value = *, left = Treenode (value = +, left = 3, right = 2), right = Treenode (value = +, left = 4, right = 5))
def evaluate_tree(node):
    if not node:
        raise ValueError("Cannot evaluate an empty tree.")
    
    # Check if it's a binary operator
    if node.value in BINARY_OPERATORS:
        left_val = evaluate_tree(node.left)
        right_val = evaluate_tree(node.right)
        return BINARY_OPERATORS[node.value](left_val, right_val)
    
    # Check if it's a unary operator
    elif node.value in UNARY_OPERATORS:
        right_val = evaluate_tree(node.right)  # Typically applies to right child
        return UNARY_OPERATORS[node.value](right_val)  # Correct unary application
    
    # Check if it's a variable
    elif node.value in VARIABLES_MAP:
        return VARIABLES_MAP[node.value]  # Lookup the variable value
    
    # Check if it's a numeric constant or coefficient
    elif isinstance(node.value, (int, float)):
        return node.value  # Return as-is for numeric leaf nodes
    
    # If none of the above, it's an error
    else:
        raise ValueError(f"Invalid node value: {node.value}")

def random_initial_tree(depth, maxdepth, variables, binary_operators, unary_operators):
    if depth == maxdepth:  # Add a variable until they are all chosen, if yes add a number
        if len(variables):
            var = random.choice(variables)
            leaf = TreeNode(var)
            leaf.coefficient = compute_coefficient()
            variables.remove(var)
        else:
            leaf = TreeNode(compute_coefficient())
            leaf.coefficient = 1
        return leaf
    
    elif depth == maxdepth - 1: # Add a unary operator
        node = TreeNode(None)
        node.right = random_initial_tree(depth + 1, maxdepth, variables, binary_operators, unary_operators)
        node.left = None
        if random.choice([0, 1]): # 50% chance of invariant unary operator, 50% chance of any of the other unary operators
            node.value = ""
        else:
            available_unary = [op for op in unary_operators if are_compatible(op, VARIABLES_MAP[node.right.value] if node.right.value in VARIABLES_MAP else int(node.right.value))]
            node.value = random.choice(available_unary) # If a choice of a variant unary operator was made, choose a random variant from all the possible ones
        return node
    
    else: # Add a binary operator
        node = TreeNode(None)
        node.left = random_initial_tree(depth + 1, maxdepth, variables, binary_operators, unary_operators)
        node.right = random_initial_tree(depth + 1, maxdepth, variables, binary_operators, unary_operators)
        available_binary = [op for op in binary_operators if are_compatible(op, evaluate_tree(node.right), evaluate_tree(node.left))]
        node.value = random.choice(available_binary) # Choose a random binary operator from all the possible ones
        return node


### printing functions

In [74]:
def print_tree(node: TreeNode, is_root: bool = True):
    """
    Prints the symbolic representation of the tree with the names of the variables
    
    Args:
        node: TreeNode
        is_root: bool
    """
    if not node:
        return

    # Add parentheses around subexpressions unless it's the root
    if not is_root:
        print("(", end="")

    # Traverse the left child
    if node.left:
        print_tree(node.left, is_root=False)

    # Print the current node's value
    print(node.value, end=" ")

    # Traverse the right child
    if node.right:
        print_tree(node.right, is_root=False)

    # Close parentheses if not the root
    if not is_root:
        print(")", end="") 


def print_tree_values(node: TreeNode, is_root: bool = True):
    """ 
    Prints the symbolic representation of the tree with the values of the variables
    
    Args:
        node: TreeNode
        is_root: bool
    """
    if not node:    
        return

    # Add parentheses around subexpressions unless it's the root
    if not is_root:
        print("(", end="")

    # Traverse the left child
    if node.left:
        print_tree_values(node.left, is_root=False)

    # Print the current node's value
    print(VARIABLES_MAP[node.value] if node.value in VARIABLES_MAP else node.value, end=" ")

    # Traverse the right child
    if node.right:
        print_tree_values(node.right, is_root=False)

    # Close parentheses if not the root
    if not is_root:
        print(")", end="") 


def print_expr(node):
    """
    Prints the symbolic representation of the tree with the names of the variables inside an expression
    """
    print_tree(node) 
    print("= y")

def print_expr_values(node):
    """
    Prints the symbolic representation of the tree with the values of the variables inside an expression
    """
    print_tree_values(node) 
    print(" = ", end="")
    print(evaluate_tree(node))

In [75]:
def generate_initial_solution():
    variables = list(VARIABLES_MAP.keys())
    n_variables = len(variables)
    n_leaves = int(2 ** np.ceil(np.log2(n_variables)))
    n_actual_leaves = n_leaves * 2
    binary_operators = list(BINARY_OPERATORS.keys())
    unary_operators = list(UNARY_OPERATORS.keys())
    unary_operators.remove("")
    max_depth = np.log2(n_actual_leaves)

    while True:
        variables = list(VARIABLES_MAP.keys())
        root = random_initial_tree(0, max_depth, variables, binary_operators, unary_operators)
        try:
            print_expr(root)
            if validate_tree(root):
                evaluate_tree(root)
                tree = Tree(root, max_depth)
                return tree
        except:
            pass
            


## steps
- Generate random tree
    - we need each variable at least once 
    - each variable has exactly one coefficient chosen as a random float number in the range [?, ?]
    - each variable has exactly one unary operator
    - unary operator is chosen as: 50% chance of "" (i.e. no change to the variable), 50% chance of choosing among all other unary operators
        - check if the unary operator is appliable to the variable ->
            ```
            leaves_map = {}
            for e in leaves:
                available_unary_operators = [op for op in list(UNARY_OPERATORS.keys()) if op.is_applicable(e)]
                chosen_unary_operator = 50% chance of "" (i.e. no change to the variable), 50% chance of choosing among [available_unary_operators]
                leaves_map[e] = [chosen_unary_operator]
            # leaves = [-2, 3]
            # leaves_map = {-2: square, 3: log}
            for e in leaves:
                node = leaves_map[e]
                node.left = null
                node.right = e
                # insert node to tree
            ```
    - number of leaves = nearest power of two greater than keys.length()
    - number of actual leaves = [number of leaves] * 2
    - number of coefficients = [number of leaves] - keys.length()
    - number of binrary operators = total number of nodes in  tree with [number of leaves] leaves - [number of leaves]]
    - validate tree
    - if valid, return tree
    - else, ?
- Example:
    - x.length() = 3
    - number of leaves = 4
    - number of actual leaves = 8
    - number of coefficients = 1
    - number of operands = 3

    ```bash
                    +
            /                  \
            *                    +
        /      \           /        \
      u        1          1          u
    /   \    /   \      /   \       /  \
    nul  *  nul   *    nul    *     nul *
    ```


In [81]:
random_tree = generate_initial_solution()
print_expr(random_tree.root)
if validate_tree(random_tree.root):
    print("valid")
    print_expr_values(random_tree.root)

((sqrt (X_2 ))^ ( (X_1 )))/ ((tan (X_0 ))/ (sqrt (7 )))= y
((sqrt (X_2 ))^ ( (X_1 )))/ ((tan (X_0 ))/ (sqrt (7 )))= y
valid
((sqrt ([3 4 5] ))^ ( ([2 3 4] )))/ ((tan ([1 2 3] ))/ (sqrt (7 ))) = [   5.09645214   -9.6867846  -464.01534089]


In [78]:
def mse(x,y):
    return (x-y)**2

In [79]:
# initial_solution = TreeNode(operators[random.randint(0,num_operators)])
# print(initial_solution.value)
# initial_solution.left = TreeNode(x[0])
# initial_solution.right = TreeNode(random.randint(1,10))
# print(initial_solution.right.value)
# print(evaluate_tree(initial_solution,{}))

# # y = x+ n
# # mse(evaluate_tree(initial_solution, {}), y[0])
# # tree
# #   operator
# #       |
# #      / \
# #    x    n

# for i in range(200):
#     sol = TreeNode(initial_solution.value)
#     sol.left = initial_solution.left
                                
#     # mutation
#     sol.value = operators[random.randint(0,num_operators)]
#     sol.right =  TreeNode(random.randint(1,10))

#     ev = evaluate_tree(sol, {})
#     # print(ev)
#     if mse(evaluate_tree(initial_solution,{}), y[0]) > mse(ev,y[0]):
#         initial_solution.value = sol.value
#         initial_solution.right = sol.right
#         print("found better solution")
#         print(mse(evaluate_tree(sol, {}),y))


# print(evaluate_tree(initial_solution,{}))
# print(f"{initial_solution.left.value} {initial_solution.value} {initial_solution.right.value}")
    

